In [ ]:
from pathlib import Path
import pymupdf
import re
import json


In [2]:
def extract_from_intmemo(pdf_path: Path) -> dict[str, str]:
    if not pdf_path.exist():
        raise FileNotFoundError(f"No existe el archivo: {pdf_path}")
    if not pdf_path.is_file():
        raise IsADirectoryError(f"La ruta no corresponde a un archivo: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Tipo de archivo no valido: {pdf_path}")
    with pymupdf.open(pdf_path) as document:
        pages_text: list[str] = []
        for page 


    

SyntaxError: invalid syntax (1916126095.py, line 10)

In [ ]:
def extract_from_report(pdf_path: Path) -> dict[str, str]:
    if not pdf_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {pdf_path}")
    if not pdf_path.is_file():
        raise IsADirectoryError(f"La ruta no corresponde a un archivo: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Tipo de archivo no valido: {pdf_path}")
    with pymupdf.open(pdf_path) as document:
        if len(document) != 1:
             raise ValueError(
                 f"Se esperaba un informe de una sola página, pero contiene {len(document)} páginas: {pdf_path}"
                 )
        
        report_data: dict[str, str] = {}
        page = document[0]
        for widget in page.widgets() or []:
            field_name = widget.field_name
            field_value = widget.field_value or ""
                    
            if field_name == "Texto":
                field_value == page.get_text(
                    "text",
                    clip = widget.rect,
                    sort=True
                )
            report_data[field_name] = field_value
    return report_data



    

In [34]:
def normalize_report(raw_data: dict[str, str]) -> dict[str, str]: 
    normalized_data: dict[str, str] = {}
    field_mapping: dict[str, str] = {
        "Servicio": "service",
        "fecha": "date",
        "puesto": "position",
        "Hora": "time",
        "Asunto": "subject",
        "Comentarios": "comments",
        "Vigilante": "author",
        "Informe": "description",
        "Texto": "corpus",
        "Texto3": "report_number"

    }
    for raw_field_name, raw_field_value in raw_data.items():
        normalized_field_name = field_mapping.get(raw_field_name)
        if normalized_field_name is None:
            continue
        cleaned_value = (
            raw_field_value
            .replace("\r\n", "\n")
            .replace("\r", "\n")
            .strip()
            .lower()
        )
        normalized_data[normalized_field_name] = cleaned_value

    return normalized_data

In [75]:
def extract_annex_number(subject: str) -> str | None:
    known_variations = ['anaexo','aniexo', 'annxo', 'anexos']
    for variation in known_variations:
        subject = subject.replace(variation, "anexo")
    match = re.search(r"anexo\D*(\d+)", subject, re.IGNORECASE)
    return match.group(1) if match else None

In [90]:
def validate_report_data(report: dict[str, str]) -> None:
    required_fields = ["corpus","time","date", "service"]
    missing_fields = [field for field in required_fields if not report.get(field, "").strip()]
    if missing_fields:
        missing_fields_text = ", ".join(missing_fields)
        raise ValueError(f'Faltan campos requeridos o estan vacios: {missing_fields_text}')
    

In [88]:
def process_report(pdf_path: Path) -> dict[str, str]:
    raw_report = extract_from_report(pdf_path)
    normalized_report = normalize_report(raw_report)
    validate_report_data(normalized_report)
    
    subject = normalized_report.get('subject', "")
    annex_number = extract_annex_number(subject)

    normalized_report["source_file"] = pdf_path.name
    normalized_report['document_type'] = "report"
    normalized_report['annex_number'] = annex_number
    
    return normalized_report

In [41]:
path = Path(r'RAW\ANEXO Nº 53 CIRCURLAR EN BICICLETA. S. TECNICAS.pdf')
report2 = process_report(path)

Informe válido
{'Servicio': 'OCEANOGRAFICO ', 'fecha': '05/04/2024', 'Puesto': 'CENTRO CONTROL', 'Hora': '10:04', 'Asunto': 'ANEXO Nº 53. BICICLETA CIRCULANDO POR SALAS TECNICAS ', 'Texto': '\r POR  MEDIO DE LA PRESENTE  LE  INFORMO  QUE:\r\rQUE SIENDO LAS 10:04 H. SE  OBSERVA POR CÁMARAS A EDUARDO NOGUÉS, CIRCULAR EN  BICICLETA POR  SALAS  TECNICAS, DESDE SU ACCESO POR PORTON AGORA, BAJANDO LA RAMPA A ZONA TECNICA-OCEANOS-A ZONA MANTENIMIENTO.Y REGRESO.\r\rSE  INFORMA DE LA  INCIDENCIA  A SUPERVISION  LUIS  COUQUE Y AL DIRECTOR  DE SEGURIDAD ALFONSO IGLESIAS QUE INDICA CONFECCIÓN DEL PRESENTE ANEXO.\r\rSE ADJUNTAN CAPTURAS.\r\r\rLO QUE SE COMUNICA PARA CONOCIMIENTO.\r\r', 'Comentarios': ' SE INFORMA AL DIRECTOR DE SEGURIDAD\r                                                                                                                                                                                                                                                                        

In [50]:
for key, value in report2.items():
    print(f'{key}: {value}')

service: oceanografico
date: 05/04/2024
time: 10:04
subject: anexo nº 53. bicicleta circulando por salas tecnicas
corpus: por  medio de la presente  le  informo  que:

que siendo las 10:04 h. se  observa por cámaras a eduardo nogués, circular en  bicicleta por  salas  tecnicas, desde su acceso por porton agora, bajando la rampa a zona tecnica-oceanos-a zona mantenimiento.y regreso.

se  informa de la  incidencia  a supervision  luis  couque y al director  de seguridad alfonso iglesias que indica confección del presente anexo.

se adjuntan capturas.


lo que se comunica para conocimiento.
comments: se informa al director de seguridad
author: emilio ramirez
description: 
source_file: ANEXO Nº 53 CIRCURLAR EN BICICLETA. S. TECNICAS.pdf
document_type: report
annex_number: 53


In [84]:
def process_reports_folder(
    folder_path: Path,
) -> tuple[list[dict[str, str]], list[dict[str, str]]]: 
    processed_reports: list[dict[str, str]] = []
    processing_errors: list[dict[str, str]] = []
    if not folder_path.exists():
        raise FileNotFoundError(
            f'no existe la carpeta {folder_path}'
        )
    if not folder_path.is_dir():
        raise NotADirectoryError(
            f'La ruta no corresponde a un directorio: {folder_path}'
        )
    pdf_files = sorted(folder_path.glob("*.pdf"))
    if not pdf_files:
        raise ValueError(f'No se encontraron archivos pdf en : {folder_path}')
    for pdf_file in pdf_files:
        try:
            processed_report = process_report(pdf_file)
            processed_reports.append(processed_report)

        except ValueError as error:
            processing_errors.append({
            "source_file": pdf_file.name,
            "error": str(error),
    })
    return processed_reports, processing_errors

In [96]:
def save_jsonl(records: list[dict[str, str]],
               output_path: Path,
               ) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as file:
        for record in records:
            json_line = json.dumps(
                record,
                ensure_ascii=False,
            )
            file.write(json_line + "\n")

In [97]:
reports, errors = process_reports_folder(Path("RAW"))

print(f"Informes procesados: {len(reports)}")
print(f"Informes rechazados: {len(errors)}")
for error in errors:
    print(
        f"Archivo rechazado: {error['source_file']}\n"
        f"Motivo: {error['error']}\n"
    )
save_jsonl(
    reports,
    Path("data/processed/reports.jsonl"),
)

save_jsonl(
    errors,
    Path("data/processed/report_errors.jsonl"),
)

Informes procesados: 6
Informes rechazados: 1
Archivo rechazado: ANEXO Nº 59 TRABAJO TÉCNICO SISTEMAS.pdf
Motivo: Faltan campos requeridos o estan vacios: time



In [98]:
print("Informes guardados:", len(reports))
print("Errores guardados:", len(errors))

Informes guardados: 6
Errores guardados: 1
